In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import sympy as sp
from IPython.display import HTML

# **Part 1: The Wave Equation**
We solve the wave equation
$$
\partial_t^2 u - c^2 \partial_x^2 u = f(t,x), \\
u(0,x) = u^0(x), \\
\partial_t u(0,x) = v^0(x)
$$
with periodic boundary conditions
$$
u(-L) = u(L).
$$



### Question 1
Implement the finite difference scheme above for the wave equation with periodic boundary conditions.

In [ ]:
def solve_wave_equation_M(M):
    """Same as solve_wave_equation() but with variable M."""
    L = 10
    a = -L
    b = L
    c = 1
    T = L / (2 * c)
 
    h = (b - a) / M
    N = int(c * T / h)
    tau = T / N
 
    xs = [a + i * h for i in range(M + 1)]
    ts = [n * tau for n in range(N + 1)]
    
    u_0 = lambda x: np.exp(-(x**2))
    v_0 = lambda x: 2 * c * x * np.exp(-(x**2))
 
    U = np.zeros((M + 1, N + 1))
    f = np.zeros((M + 1, N + 1))
 
    # Initial condition n=0
    for i in range(M + 1):
        U[i, 0] = u_0(xs[i])
 
    # First time step n=1 (interior)
    for i in range(1, M):
        U[i, 1] = (U[i, 0]
                   + tau * v_0(xs[i])
                   + (c**2 * tau**2) / (2 * h**2) * (U[i+1, 0] - 2*U[i, 0] + U[i-1, 0])
                   + (tau**2 / 2) * f[i, 0])
 
    # Periodic boundary at n=1
    U[0, 1] = (U[0, 0]
               + tau * v_0(xs[0])
               + (c**2 * tau**2) / (2 * h**2) * (U[1, 0] - 2*U[0, 0] + U[M, 0])
               + (tau**2 / 2) * f[0, 0])
    U[M, 1] = U[0, 1]
 
    # Main time loop
    for n in range(1, N):
        for i in range(1, M):
            U[i, n+1] = (2*U[i, n] - U[i, n-1]
                         + (c**2 * tau**2) / (h**2) * (U[i+1, n] - 2*U[i, n] + U[i-1, n])
                         + tau**2 * f[i, n])
        U[0, n+1] = (2*U[0, n] - U[0, n-1]
                     + (c**2 * tau**2) / (h**2) * (U[1, n] - 2*U[0, n] + U[M, n])
                     + tau**2 * f[0, n])
        U[M, n+1] = U[0, n+1]
 
    return U, xs, ts

: 

### Question 2
Test your code with the method of manufactured solutions.

Let $L = 10$, $c = 1$, $u(t,x) = \exp(-(x-ct)^2)$.
Compute the corresponding $f$, $u^0$, and $v^0$ such that $u$ solves the PDE.
Run until final time $T := L/(2c) = N\tau$.
        
Plot $U_i^N$ versus $u(t_N,x_i)$ at the grid points $x_i$ for the highest refinement level.

In [ ]:
def u_exact(x, t):
  c=1
  return np.exp(-(x-c*t)**2)

def test_wave_equation():
  L = 10
  a = -L
  b = L
  c = 1
  T = L/(2*c)
  M = 200

  u_0 = lambda x: np.exp(-(x**2))
  v_0 = lambda x: 2*c*x*np.exp(-(x**2))

  h = (b - a) / M 

  N = int(c * T / h) 
  tau = T / N 

  xs = [a + i * h for i in range(M + 1)] # M+1 spatial points from a to b
  ts = [n * tau for n in range(N + 1)] # N+1 time points from 0 to T

  U = solve_wave_equation_M(200)

  if isinstance(U, tuple):
    U = U[0]

  # set the initial plot
  fig, ax = plt.subplots()
  U_max = np.max(U)
  U_min = np.min(U)
  abs_U_max = np.max(np.abs(U))
  y_min = U_min - 0.1 * abs_U_max
  y_max = U_max + 0.1 * abs_U_max
  ax.set(
      ylim=[y_min, y_max], xlabel="x", ylabel="u(x,t)", title="t = {}".format(ts[0])
  )
  line = ax.plot(xs, U[:, 0])[0]

    # update the plot each frame
  def update(frame):
      line.set_ydata(U[:, frame])
      ax.set(title="t = {}".format(ts[frame]))
      return line

  # create the animation
  ani = animation.FuncAnimation(fig=fig, func=update, frames=U.shape[1], interval=30)
  plt.close()
  return HTML(ani.to_jshtml())
test_wave_equation()

### Question 3
Let
	      \begin{equation}
		      e_k := \max_{0\leq i \leq M} |U_i^N - u(t_N,x_i)|
	      \end{equation}
	      denote the discrete error at the final time for mesh refinement level $k$, where we recall that $M = 2^k$.
	      Create a log--log plot of $e_k$ versus the mesh size $h_k := 2L/M = L / 2^{k-1}$ for $k = 10,11,12$.

In [ ]:
def compute_error(k):
    L = 10
    T = L / 2   # c=1, T = L/(2c)
 
    M = 2**k
    h_k = 2 * L / M   # = L / 2^{k-1}
 
    U, xs, _ = solve_wave_equation_M(M)
 
    # e_k = max_{0<=i<=M} |U^N_i - u(t_N, x_i)|
    U_exact = np.array([u_exact(x, T) for x in xs])
    e_k = np.max(np.abs(U[:, -1] - U_exact))
 
    return h_k, e_k
 
 
# --- log-log convergence plot for k = 10, 11, 12 ---
ks = [10, 11, 12]
hs = []
es = []
print("h\t\tE_M")
for k in ks:
    h_k, e_k = compute_error(k)
    hs.append(h_k)
    es.append(e_k)
    print(f"{h_k:.6e}, {e_k:.6e}")
 
hs = np.array(hs)
es = np.array(es)
 
# reference slope-2 line
ref = es[0] * (hs / hs[0])**2
 
plt.figure(figsize=(7, 5))
plt.loglog(hs, es,  'o-',  color='steelblue', linewidth=2, markersize=8, label=r'$e_k$')
plt.loglog(hs, ref, 'k--', linewidth=1.5, label=r'$O(h^2)$ reference')
 
for h, e, k in zip(hs, es, ks):
    plt.annotate(f'k={k}', xy=(h, e), xytext=(6, 4),
                 textcoords='offset points', fontsize=10)
 
plt.xlabel(r'Mesh size $h_k = 2L/M$', fontsize=13)
plt.ylabel(r'Max error $e_k$', fontsize=13)
plt.title(r'Log-log convergence plot: $e_k$ vs $h_k$', fontsize=13)
plt.legend(fontsize=11)
plt.grid(True, which='both', ls='--', alpha=0.4)
plt.tight_layout()
plt.show()

### Question 4
Make an animation of your approximate solution $U_i^n$.

In [ ]:
test_wave_equation()

### Question 5
Now set $f(t,x) = 0$ and $v^0 = 0$.
	      Experiment with different periodic initial conditions $u^0$.
	      Take snapshots of your simulations and create animations over the circle in 3d.

In [ ]:
def animate_over_circle(t_values,s_values,u_values):
    fig = plt.figure()
    ax = fig.add_subplot(projection='3d')
    ax.set_title('t = {:.2f}'.format(t_values[0]))
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_zlabel('u')
    z_min = min(min(u_value) for u_value in u_values)
    z_min -= 0.1 if abs(z_min) < 0.1 else 0.1 * abs(z_min)
    z_max = max(max(u_value) for u_value in u_values)
    z_max += 0.1 if abs(z_max) < 0.1 else 0.1 * abs(z_max)
    ax.set_zlim(z_min,z_max)
    L = -s_values[0]
    r = L
    
    
    xs=[L*np.cos(s/r)for s in s_values]
    ys=[L*np.sin(s/r)for s in s_values]
    
    line = ax.plot(xs, ys, u_values[0])[0]
    def animate(frame):
        ax.set_title('t = {:.2f}'.format(t_values[frame]))
        line.set_data_3d(xs, ys, u_values[frame])
        return line
    anim = animation.FuncAnimation(fig,animate,frames=len(u_values),interval=30)
    plt.close()
    video = HTML(anim.to_html5_video())
    return video

L=10
a = -L
b = L
c = 1
T = L / (2*c)
k=12
M = 2**k
h = (b - a) / (M)
N = int(c * T / h)
tau = T / (N)

xs = [a + i * h for i in range(M + 1)]
ts = [n * tau for n in range(N + 1)]

U, x ,y = solve_wave_equation_M(M)
u_values = [U[:,n]for n in range(U.shape[1])]
animate_over_circle(y,x,u_values)

In [ ]:
def solve_dispersive_pde_equation(
    L=10, c=1, k=12,
    boundary="periodic",
    f=lambda x, t: 0,
    u_0=lambda x: np.exp(-(x**2)),
    v_0=lambda x: 2*c*x*np.exp(-(x**2)),
    g=lambda u: np.zeros_like(u),
    T = L / (2*c)
):
    a = -L
    b = L
    T = L / (2*c)
    M = 2**k

    bd = boundary

    h = (b - a) / M

    cfl = .25

    N = int(c * T / (cfl*h))
    tau = T / N

    xs = np.linspace(a, b, M + 1)
    ts = np.linspace(0, T, N + 1)

    U = np.zeros((M + 1, N + 1))

    U[:, 0] = u_0(xs)

    # boundary conditions at t_0
    U[0, 0] = u_0(xs[0])
    U[M, 0] = u_0(xs[M])  # U[0, 0]

    # At t_1
    U[1:M, 1] = (
        U[1:M, 0]
        + tau * v_0(xs[1:M])
        + tau**2 * c**2 / (2 * h**2) * (U[2:M+1, 0] - 2 * U[1:M, 0] + U[0:M-1, 0])
        + tau**2 / 2 * f(xs[1:M], ts[0])
        - tau**2 / 2 * g(U[1:M, 0])
    )

    # absorbing boundary conditions at t_1
    if boundary == "absorbing":
        U[0, 1] = U[1, 0]
        U[M, 1] = U[M-1, 0]

    # periodic boundary conditions at t_1
    if boundary == "periodic":
        U[0, 1] = (
            U[0, 0] + tau * v_0(xs[0])
            + tau**2 * c**2 / (2 * h**2) * (U[1, 0] - 2 * U[0, 0] + U[M - 1, 0])
            + tau**2 / 2 * f(xs[0], ts[0]) - tau**2 / 2 * g(U[0, 0])
        )
        U[M, 1] = U[0, 1]

    coeff = tau**2 * c**2 / h**2

    # explicit timestepping
    for n in range(1, N):
        U[1:M, n + 1] = (
            2 * U[1:M, n]
            - U[1:M, n - 1]
            + coeff * (U[2:M+1, n] - 2 * U[1:M, n] + U[0:M-1, n])
            + tau**2 * f(xs[1:M], ts[n])
            - tau**2 * g(U[1:M, n])
        )

        # absorbing boundary conditions at t_1
        if boundary == "absorbing":
            U[0, n+1] = U[1, n]
            U[M, n+1] = U[M-1, n]

        if boundary == "periodic":
            U[0, n + 1] = (
                2 * U[0, n]
                - U[0, n - 1]
                + coeff * (U[1, n] - 2 * U[0, n] + U[M - 1, n])
                + tau**2 * f(xs[0], ts[n]) - tau**2 * g(U[0, n])
            )
            U[M, n + 1] = U[0, n + 1]

    return U, xs, ts

In [ ]:
x, t= sp.symbols('x t')

c = sp.Integer(1)
omega = sp.Rational(1, 2)

#HINT: use sp for the functions, for example, for sin, use sp.sin

u_exact = sp.cos(sp.pi*x+sp.sqrt((sp.pi)**2+1)*t)

dtt_u_exact = sp.diff(u_exact, t, 2)

d_xx_u_exact = sp.diff(u_exact, x, 2)

f = dtt_u_exact-c**2*d_xx_u_exact+u_exact

sp.trigsimp(sp.simplify(f))

# Hint: the output here should be 0!


In [ ]:
L = 1
c = 1
g = lambda u : u
T = (2*np.pi) / sp.sqrt(np.pi**2+1)

x, t = sp.symbols('x t')
u = sp.cos(sp.pi * x + sp.sqrt(sp.pi**2 + 1)*t)
du_dt = sp.diff(u, t)
u0 = u.subs(t, 0)
v0 = du_dt.subs(t, 0)
u0 = sp.lambdify(x, u0, 'numpy')
v0 = sp.lambdify(x, v0, 'numpy')
f  = sp.lambdify((x, t), f, 'numpy')
u_exact = sp.lambdify((x, t), u, 'numpy')
g = lambda u: u

def compute_errors(
        L=L,
        c=c,
        g = g,
        u0 = u0,
        v0 = v0,
        f = f,
        u_exact = u_exact,
        T=T
    ):
    ks = np.array([10,11,12])
    Ms = 2 ** ks
    hs = 2 * L / Ms
    errors = np.zeros(len(ks))


    for j in range(len(ks)):
        k = ks[j]
        U_approx, xs, ts = solve_dispersive_pde_equation(L, c, k, "periodic",f,u0,v0,g,T)
        u_final = u_exact(np.array(xs), ts[-1]) 
        errors[j] = np.max(np.abs(U_approx[:, -1] - u_final))

    fig, ax = plt.subplots()
    ax.loglog(hs, errors, 'o-', label='error')
    ax.set_xlabel('h_k')
    ax.set_ylabel('e_k')
    ax.set_title('Final-time error vs mesh size')
    ax.grid(True, which='both')
    ax.legend()
    plt.show()

    print(hs)
    print(errors)

    slope = ( (np.log(errors[-1]) - np.log(errors[0])) / (np.log(hs[-1]) - np.log(hs[0])) )
    print(f"Slope of log-log plot: + {slope}")



    return

compute_errors(L,c,g,u0,v0,f,u_exact,T)

In [ ]:
def solve_dispersive_pde_equation(
    L=100, c=1, k=12,
    boundary="periodic",
    f=lambda x, t: 0,
    omega = .99,
    u_0=lambda x: 4*np.atan((np.cos(omega*t)*np.sqrt(1-omega**2))/omega),
    v_0=lambda x: 0,
    g=lambda u: np.sin(u),
    T = 2*np.pi / (np.sqrt(np.pi**2+1))
):
    a = -L
    b = L
    T = 2*np.pi / (np.sqrt(np.pi**2+1))
    M = 2**k

    bd = boundary

    h = (b - a) / M

    cfl = .9

    N = int(c * T / (cfl*h))
    tau = T / N

    xs = np.linspace(a, b, M + 1)
    ts = np.linspace(0, T, N + 1)

    U = np.zeros((M + 1, N + 1))

    U[:, 0] = u_0(xs)

    # boundary conditions at t_0
    U[0, 0] = u_0(xs[0])
    U[M, 0] = u_0(xs[M])  # U[0, 0]

    # At t_1
    U[1:M, 1] = (
        U[1:M, 0]
        + tau * v_0(xs[1:M])
        + tau**2 * c**2 / (2 * h**2) * (U[2:M+1, 0] - 2 * U[1:M, 0] + U[0:M-1, 0])
        + tau**2 / 2 * f(xs[1:M], ts[0])
        - tau**2 / 2 * g(U[1:M, 0])
    )

    # absorbing boundary conditions at t_1
    if boundary == "absorbing":
        U[0, 1] = U[1, 0]
        U[M, 1] = U[M-1, 0]

    # periodic boundary conditions at t_1
    if boundary == "periodic":
        U[0, 1] = (
            U[0, 0] + tau * v_0(xs[0])
            + tau**2 * c**2 / (2 * h**2) * (U[1, 0] - 2 * U[0, 0] + U[M - 1, 0])
            + tau**2 / 2 * f(xs[0], ts[0]) - tau**2 / 2 * g(U[0, 0])
        )
        U[M, 1] = U[0, 1]

    coeff = tau**2 * c**2 / h**2

    # explicit timestepping
    for n in range(1, N):
        U[1:M, n + 1] = (
            2 * U[1:M, n]
            - U[1:M, n - 1]
            + coeff * (U[2:M+1, n] - 2 * U[1:M, n] + U[0:M-1, n])
            + tau**2 * f(xs[1:M], ts[n])
            - tau**2 * g(U[1:M, n])
        )

        # absorbing boundary conditions at t_1
        if boundary == "absorbing":
            U[0, n+1] = U[1, n]
            U[M, n+1] = U[M-1, n]

        if boundary == "periodic":
            U[0, n + 1] = (
                2 * U[0, n]
                - U[0, n - 1]
                + coeff * (U[1, n] - 2 * U[0, n] + U[M - 1, n])
                + tau**2 * f(xs[0], ts[n]) - tau**2 * g(U[0, n])
            )
            U[M, n + 1] = U[0, n + 1]

    return U, xs, ts
x, t= sp.symbols('x t')

c = sp.Integer(1)
omega = sp.Rational(99,100)


u_exact = 4*sp.atan((sp.cos(omega*t)*sp.sqrt(1-omega**2))/(omega*sp.cosh(x*sp.sqrt(1-omega**2))))

dtt_u_exact = sp.diff(u_exact, t, 2)

d_xx_u_exact = sp.diff(u_exact, x, 2)

f = dtt_u_exact-c**2*d_xx_u_exact+sp.sin(u_exact)

sp.trigsimp(sp.simplify(f))

L = 1
c = 1
g = lambda u : u
T = (2*np.pi) / sp.sqrt(np.pi**2+1)

x, t = sp.symbols('x t')
u = sp.cos(sp.pi * x + sp.sqrt(sp.pi**2 + 1)*t)
du_dt = sp.diff(u, t)
u0 = u.subs(t, 0)
v0 = du_dt.subs(t, 0)
u0 = sp.lambdify(x, u0, 'numpy')
v0 = sp.lambdify(x, v0, 'numpy')
f  = sp.lambdify((x, t), f, 'numpy')
u_exact = sp.lambdify((x, t), u, 'numpy')
g = lambda u: u

def compute_errors(
        L=L,
        c=c,
        g = g,
        u0 = u0,
        v0 = v0,
        f = f,
        u_exact = u_exact,
        T=T
    ):
    ks = np.array([10,11,12])
    Ms = 2 ** ks
    hs = 2 * L / Ms
    errors = np.zeros(len(ks))
    omega=.99


    for j in range(len(ks)):
        k = ks[j]
        U_approx, xs, ts = solve_dispersive_pde_equation(L, c, k, "periodic",f,omega,u0,v0,g,T)
        u_final = u_exact(np.array(xs), ts[-1]) 
        errors[j] = np.max(np.abs(U_approx[:, -1] - u_final))

    fig, ax = plt.subplots()
    ax.loglog(hs, errors, 'o-', label='error')
    ax.set_xlabel('h_k')
    ax.set_ylabel('e_k')
    ax.set_title('Final-time error vs mesh size')
    ax.grid(True, which='both')
    ax.legend()
    plt.show()

    print(hs)
    print(errors)

    slope = ( (np.log(errors[-1]) - np.log(errors[0])) / (np.log(hs[-1]) - np.log(hs[0])) )
    print(f"Slope of log-log plot: + {slope}")



    return

compute_errors(L,c,g,u0,v0,f,u_exact,T)